In [1]:
import json
import joblib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
    PrecisionRecallDisplay
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
PROCESSED_DIR = Path("data/processed")
MODELS_DIR = Path("artifacts/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

train_path = PROCESSED_DIR / "train_scorer.csv"
val_path = PROCESSED_DIR / "val_scorer.csv"
test_path = PROCESSED_DIR / "test_scorer.csv"

In [6]:
train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print(train_df.shape, val_df.shape, test_df.shape)


(129556, 16) (16195, 16) (16195, 16)


In [15]:
AA_LIST = list("ACDEFGHIKLMNPQRSTVWY")

def aa_fraction_features(seq):
    counts = Counter(seq)
    n = len(seq)
    return {f"frac_{aa}": counts.get(aa, 0) / n for aa in AA_LIST}

def extra_sequence_features(seq):
    n = len(seq)
    counts = Counter(seq)

    basic = counts.get("K", 0) + counts.get("R", 0)
    acidic = counts.get("D", 0) + counts.get("E", 0)
    aromatic = counts.get("F", 0) + counts.get("W", 0) + counts.get("Y", 0)
    aliphatic = counts.get("A", 0) + counts.get("V", 0) + counts.get("I", 0) + counts.get("L", 0) + counts.get("M", 0)
    polar = counts.get("S", 0) + counts.get("T", 0) + counts.get("N", 0) + counts.get("Q", 0)
    
    return {
        "basic_count": basic,
        "acidic_count": acidic,
        "aromatic_frac": aromatic / n,
        "aliphatic_frac": aliphatic / n,
        "polar_frac": polar / n,
        "proline_frac": counts.get("P", 0) / n,
        "glycine_frac": counts.get("G", 0) / n,
        "cysteine_frac": counts.get("C", 0) / n,
        "lys_arg_ratio": counts.get("K", 0) / max((counts.get("R", 0) + 1), 1),
    }

In [17]:
BASE_NUMERIC_COLS = [
    "length",
    "charge",
    "hydrophobicity",
    "frac_basic",
    "frac_acidic",
    "cysteine_count",
]

def make_feature_df(df):
    rows = []
    for _, row in df.iterrows():
        seq = row["sequence"]
        feats = {}

        for col in BASE_NUMERIC_COLS:
            feats[col] = row[col]

        feats.update(extra_sequence_features(seq))
        feats.update(aa_fraction_features(seq))
        rows.append(feats)

    return pd.DataFrame(rows)

In [18]:
X_train = make_feature_df(train_df)
X_val = make_feature_df(val_df)
X_test = make_feature_df(test_df)

y_train = train_df["label"].values
y_val = val_df["label"].values
y_test = test_df["label"].values

print(X_train.shape, X_val.shape, X_test.shape)
X_train.head()

(129556, 35) (16195, 35) (16195, 35)


,length,charge,hydrophobicity,frac_basic,frac_acidic,cysteine_count,basic_count,acidic_count,aromatic_frac,aliphatic_frac,...,frac_M,frac_N,frac_P,frac_Q,frac_R,frac_S,frac_T,frac_V,frac_W,frac_Y
0,43,3,0.551163,0.116279,0.046512,1,5,2,0.116279,0.441860,...,0.116279,0.000000,0.023256,0.000000,0.046512,0.069767,0.046512,0.093023,0.023256,0.046512
1,47,-2,0.602128,0.085106,0.127660,2,4,6,0.021277,0.489362,...,0.021277,0.042553,0.021277,0.000000,0.021277,0.021277,0.063830,0.085106,0.021277,0.000000
2,46,1,-0.045652,0.086957,0.065217,2,4,3,0.021739,0.347826,...,0.021739,0.021739,0.130435,0.021739,0.021739,0.173913,0.021739,0.086957,0.000000,0.000000
3,50,-4,0.386000,0.060000,0.140000,2,3,7,0.100000,0.380000,...,0.080000,0.060000,0.000000,0.060000,0.040000,0.100000,0.040000,0.020000,0.000000,0.000000
4,34,0,0.473529,0.058824,0.058824,0,2,2,0.000000,0.411765,...,0.058824,0.029412,0.117647,0.000000,0.029412,0.029412,0.088235,0.117647,0.000000,0.000000


In [19]:
logreg_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

logreg_clf.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work

In [20]:
def evaluate_classifier(model, X, y, split_name="val", threshold=0.5):
    prob = model.predict_proba(X)[:, 1]
    pred = (prob >= threshold).astype(int)

    metrics = {
        "split": split_name,
        "threshold": threshold,
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "roc_auc": roc_auc_score(y, prob),
        "pr_auc": average_precision_score(y, prob),
    }
    return metrics, prob, pred

val_metrics, val_prob, val_pred = evaluate_classifier(logreg_clf, X_val, y_val, "val")
pd.Series(val_metrics)

split             val
threshold         0.5
accuracy     0.930658
precision    0.756584
recall       0.917619
f1           0.829357
roc_auc       0.97431
pr_auc       0.905585
dtype: object

In [21]:
thresholds = np.arange(0.1, 0.91, 0.05)
rows = []

for thr in thresholds:
    m, _, _ = evaluate_classifier(logreg_clf, X_val, y_val, "val", threshold=thr)
    rows.append(m)

thr_df = pd.DataFrame(rows)
thr_df.sort_values("f1", ascending=False).head(10)

,split,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc
11,val,0.65,0.941525,0.814849,0.881977,0.847085,0.97431,0.905585
13,val,0.75,0.943933,0.848986,0.844990,0.846983,0.97431,0.905585
12,val,0.70,0.942637,0.830799,0.863484,0.846826,0.97431,0.905585
14,val,0.80,0.944242,0.868899,0.820108,0.843799,0.97431,0.905585
10,val,0.60,0.938191,0.794918,0.894082,0.841589,0.97431,0.905585
15,val,0.85,0.943316,0.890281,0.788500,0.836305,0.97431,0.905585
9,val,0.55,0.934115,0.775180,0.903161,0.834291,0.97431,0.905585
8,val,0.50,0.930658,0.756584,0.917619,0.829357,0.97431,0.905585
16,val,0.90,0.940105,0.917500,0.740417,0.819501,0.97431,0.905585
7,val,0.45,0.924792,0.733511,0.927371,0.819127,0.97431,0.905585


In [22]:
best_threshold = float(thr_df.sort_values("f1", ascending=False).iloc[0]["threshold"])
best_threshold

0.6500000000000001

In [23]:
val_metrics, val_prob, val_pred = evaluate_classifier(
    logreg_clf, X_val, y_val, "val", threshold=best_threshold
)

print(pd.Series(val_metrics))
print()
print(classification_report(y_val, val_pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_val, val_pred))

split             val
threshold        0.65
accuracy     0.941525
precision    0.814849
recall       0.881977
f1           0.847085
roc_auc       0.97431
pr_auc       0.905585
dtype: object

              precision    recall  f1-score   support

           0     0.9730    0.9549    0.9639     13221
           1     0.8148    0.8820    0.8471      2974

    accuracy                         0.9415     16195
   macro avg     0.8939    0.9184    0.9055     16195
weighted avg     0.9439    0.9415    0.9424     16195

Confusion matrix:
 [[12625   596]
 [  351  2623]]


In [ ]:
rf_clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

rf_clf.fit(X_train, y_train)


split             val
threshold         0.5
accuracy     0.974992
precision    0.934981
recall       0.928379
f1           0.931669
roc_auc      0.993901
pr_auc       0.981051
dtype: object

In [32]:

rf_val_metrics, rf_val_prob, rf_val_pred = evaluate_classifier(
    rf_clf, X_val, y_val, "val", threshold=0.5
)

pd.Series(rf_val_metrics)

split             val
threshold         0.5
accuracy     0.974992
precision    0.934981
recall       0.928379
f1           0.931669
roc_auc      0.993901
pr_auc       0.981051
dtype: object

In [33]:
thresholds = np.arange(0.1, 0.91, 0.05)
rows = []

for thr in thresholds:
    m, _, _ = evaluate_classifier(rf_clf, X_val, y_val, "val", threshold=thr)
    rows.append(m)

thr_df = pd.DataFrame(rows)
thr_df.sort_values("f1", ascending=False).head(10)

,split,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc
9,val,0.55,0.975672,0.947295,0.918628,0.932742,0.993901,0.981051
8,val,0.50,0.974992,0.934981,0.928379,0.931669,0.993901,0.981051
7,val,0.45,0.974498,0.922189,0.940484,0.931247,0.993901,0.981051
10,val,0.60,0.974745,0.957219,0.902824,0.929227,0.993901,0.981051
6,val,0.40,0.972275,0.905037,0.948554,0.926285,0.993901,0.981051
11,val,0.65,0.972893,0.964456,0.885003,0.923023,0.993901,0.981051
5,val,0.35,0.969682,0.886640,0.957297,0.920614,0.993901,0.981051
12,val,0.70,0.971164,0.973555,0.866510,0.916919,0.993901,0.981051
4,val,0.30,0.965915,0.865861,0.963685,0.912158,0.993901,0.981051
13,val,0.75,0.967644,0.981147,0.839946,0.905072,0.993901,0.981051


In [25]:
importances = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf_clf.feature_importances_
}).sort_values("importance", ascending=False)

display(importances.head(20))

,feature,importance
0,length,0.236835
25,frac_M,0.170300
7,acidic_count,0.066325
18,frac_E,0.054052
4,frac_acidic,0.050904
30,frac_S,0.029990
16,frac_C,0.029140
13,cysteine_frac,0.025283
17,frac_D,0.024521
28,frac_Q,0.023056


In [26]:
no_length_cols = [c for c in X_train.columns if c != "length"]
rf_no_len = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)
rf_no_len.fit(X_train[no_length_cols], y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [ ]:
print("Evaluating model without length feature:")
pd.Series(evaluate_classifier(rf_no_len, X_val[no_length_cols], y_val, "val", threshold=0.5)[0])    

Evaluating model without length feature:


split             val
threshold         0.5
accuracy     0.973943
precision    0.940608
recall       0.915938
f1           0.928109
roc_auc      0.993072
pr_auc       0.979123
dtype: object

In [29]:
proxy_cols = ["length", "charge", "hydrophobicity"]
rf_proxy = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)
rf_proxy.fit(X_train[proxy_cols], y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [30]:
print("Evaluating model with only most important features:")
pd.Series(evaluate_classifier(rf_proxy, X_val[proxy_cols], y_val, "val", threshold=0.5)[0])

Evaluating model with only most important features:


split             val
threshold         0.5
accuracy     0.923495
precision    0.779029
recall       0.814391
f1           0.796318
roc_auc      0.947273
pr_auc       0.871134
dtype: object

The coarse control variables used for conditioning capture part of the AMP/non-AMP distinction, but a richer sequence-composition model performs substantially better, showing that the reranker uses additional information beyond simple length, charge, and hydrophobicity.

We also tested whether the AMP scorer was relying only on the simple control variables used for generation. A random forest trained on length, approximate charge, and hydrophobicity alone performed reasonably well, but remained clearly below the full feature-based model. In contrast, removing sequence length from the full model caused only a small drop in performance. This suggests that the scorer is not driven solely by peptide length or by the three coarse conditioning proxies, but also benefits from more detailed amino-acid composition features. For our project, this is desirable: the conditioning variables remain simple and interpretable for the user, while the reranker provides a stronger sequence-level filter for selecting candidates.

In [35]:
import json
import joblib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

In [36]:
MODELS_DIR = Path("artifacts/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

final_model = rf_clf
final_model_name = "rf_amp_scorer_v1"
final_threshold = 0.5

model_path = MODELS_DIR / f"{final_model_name}.joblib"
meta_path = MODELS_DIR / f"{final_model_name}_meta.json"

joblib.dump(final_model, model_path)

metadata = {
    "model_name": final_model_name,
    "threshold": final_threshold,
    "feature_columns": list(X_train.columns),
    "notes": {
        "task": "AMP vs non-AMP scoring",
        "model_type": "RandomForestClassifier",
        "sequence_filters": "standard amino acids only, peptide length 10-50",
        "conditioning_proxies": ["length", "charge", "hydrophobicity"],
    }
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved model to:", model_path)
print("Saved metadata to:", meta_path)

Saved model to: artifacts/models/rf_amp_scorer_v1.joblib
Saved metadata to: artifacts/models/rf_amp_scorer_v1_meta.json


In [ ]:
# how to load the model and metadata later:

# def load_rf_scorer(model_path, meta_path):
#     model = joblib.load(model_path)
#     with open(meta_path, "r", encoding="utf-8") as f:
#         metadata = json.load(f)
#     return model, metadata


# rf_model, rf_meta = load_rf_scorer(model_path, meta_path)
# rf_feature_columns = rf_meta["feature_columns"]
# rf_threshold = rf_meta["threshold"]

# print(rf_meta["model_name"])
# print("Threshold:", rf_threshold)
# print("Num features:", len(rf_feature_columns))

In [37]:
AA_LIST = list("ACDEFGHIKLMNPQRSTVWY")

KYTE_DOOLITTLE = {
    "A": 1.8, "C": 2.5, "D": -3.5, "E": -3.5, "F": 2.8,
    "G": -0.4, "H": -3.2, "I": 4.5, "K": -3.9, "L": 3.8,
    "M": 1.9, "N": -3.5, "P": -1.6, "Q": -3.5, "R": -4.5,
    "S": -0.8, "T": -0.7, "V": 4.2, "W": -0.9, "Y": -1.3
}

STANDARD_AAS = set(AA_LIST)


def validate_sequence(seq: str):
    seq = seq.strip().upper()
    if len(seq) == 0:
        raise ValueError("Sequence is empty.")
    bad_chars = sorted(set(seq) - STANDARD_AAS)
    if bad_chars:
        raise ValueError(f"Sequence contains non-standard amino acids: {bad_chars}")
    return seq


def approximate_net_charge(seq: str) -> int:
    return seq.count("K") + seq.count("R") - seq.count("D") - seq.count("E")


def mean_hydrophobicity(seq: str) -> float:
    return float(np.mean([KYTE_DOOLITTLE[aa] for aa in seq]))


def aa_fraction_features(seq: str):
    counts = Counter(seq)
    n = len(seq)
    return {f"frac_{aa}": counts.get(aa, 0) / n for aa in AA_LIST}


def extra_sequence_features(seq: str):
    counts = Counter(seq)
    n = len(seq)

    basic = counts.get("K", 0) + counts.get("R", 0)
    acidic = counts.get("D", 0) + counts.get("E", 0)
    aromatic = counts.get("F", 0) + counts.get("W", 0) + counts.get("Y", 0)
    aliphatic = (
        counts.get("A", 0) + counts.get("V", 0) + counts.get("I", 0)
        + counts.get("L", 0) + counts.get("M", 0)
    )
    polar = counts.get("S", 0) + counts.get("T", 0) + counts.get("N", 0) + counts.get("Q", 0)

    return {
        "length": n,
        "charge": approximate_net_charge(seq),
        "hydrophobicity": mean_hydrophobicity(seq),
        "frac_basic": basic / n,
        "frac_acidic": acidic / n,
        "cysteine_count": counts.get("C", 0),
        "basic_count": basic,
        "acidic_count": acidic,
        "aromatic_frac": aromatic / n,
        "aliphatic_frac": aliphatic / n,
        "polar_frac": polar / n,
        "proline_frac": counts.get("P", 0) / n,
        "glycine_frac": counts.get("G", 0) / n,
        "cysteine_frac": counts.get("C", 0) / n,
        "lys_arg_ratio": counts.get("K", 0) / max(counts.get("R", 0) + 1, 1),
    }


def compute_features_for_sequence(seq: str):
    seq = validate_sequence(seq)

    feats = {}
    feats.update(extra_sequence_features(seq))
    feats.update(aa_fraction_features(seq))
    return feats

In [40]:
def score_sequence_amp_probability(seq: str, model, feature_columns):
    feat_dict = compute_features_for_sequence(seq)
    x = pd.DataFrame([feat_dict])

    missing_cols = [c for c in feature_columns if c not in x.columns]
    if missing_cols:
        raise ValueError(f"Missing required feature columns: {missing_cols}")

    x = x[feature_columns]
    prob = model.predict_proba(x)[:, 1][0]
    return float(prob)

In [41]:
def score_sequences_batch(sequences, model, feature_columns, threshold=0.5):
    rows = []
    valid_sequences = []

    for seq in sequences:
        try:
            seq = validate_sequence(seq)
            valid_sequences.append(seq)
            rows.append(compute_features_for_sequence(seq))
        except ValueError:
            continue

    if not rows:
        return pd.DataFrame(columns=[
            "sequence", "length", "charge", "hydrophobicity",
            "amp_probability", "amp_prediction",
            "frac_basic", "frac_acidic", "cysteine_count"
        ])

    X = pd.DataFrame(rows)[feature_columns]
    probs = model.predict_proba(X)[:, 1]
    preds = (probs >= threshold).astype(int)

    out = pd.DataFrame({
        "sequence": valid_sequences,
        "length": [r["length"] for r in rows],
        "charge": [r["charge"] for r in rows],
        "hydrophobicity": [r["hydrophobicity"] for r in rows],
        "frac_basic": [r["frac_basic"] for r in rows],
        "frac_acidic": [r["frac_acidic"] for r in rows],
        "cysteine_count": [r["cysteine_count"] for r in rows],
        "amp_probability": probs,
        "amp_prediction": preds,
    })

    return out.sort_values("amp_probability", ascending=False).reset_index(drop=True)